#  Notebook 3 — Modelo Sesgado
## HMDA New York 2024: Sesgo, Equidad y Explicabilidad en Hipotecas

**Autores:** Izan Cuesta Corbí · Dennis García Solera · Marcos Segurado Llopis · Jesús Cano Moya  
**Dataset:** Home Mortgage Disclosure Act (HMDA) — New York 2024 (CFPB)

---
> **Objetivo de este notebook:** Comparar y optimizar modelos de ensamble para seleccionar el mejor predictor. Evaluaremos su rendimiento, cuantificaremos su sesgo inherente hacia grupos sensibles y auditaremos sus decisiones mediante valores SHAP.

> `RANDOM_STATE = 42` fijado globalmente para reproducibilidad.

--- 

### Imports

In [1]:
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    f1_score,
    accuracy_score
)

---
---

## Conjunto de Datos

---

### Carga del dataset

In [2]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

train.head()

Train: (226665, 81)
Test:  (56667, 81)


,derived_msa-md,county_code,preapproval,loan_type,loan_purpose,lien_status,reverse_mortgage,open-end_line_of_credit,business_or_commercial_purpose,loan_amount,...,derived_race_Black or African American,derived_race_Free Form Text Only,derived_race_Joint,derived_race_Native Hawaiian or Other Pacific Islander,derived_race_Race Not Available,derived_race_White,derived_sex_Female,derived_sex_Joint,derived_sex_Male,derived_sex_Sex Not Available
0,-0.053043,-0.136364,0.0,0.0,9.666667,1.0,0.0,1.0,0.0,-0.030303,...,False,False,False,False,False,True,False,True,False,False
1,5.598696,0.772727,0.0,0.0,-0.333333,0.0,0.0,0.0,0.0,-0.303030,...,False,False,False,False,False,True,False,False,True,False
2,-1.898609,-1.318182,0.0,0.0,0.666667,1.0,0.0,0.0,0.0,-0.545455,...,False,False,False,False,False,True,False,True,False,False
3,0.414435,-0.227273,0.0,0.0,-0.333333,0.0,0.0,0.0,1.0,-0.272727,...,False,False,False,False,True,False,False,False,False,True
4,0.000000,0.363636,0.0,0.0,9.666667,0.0,0.0,0.0,1.0,0.696970,...,False,False,False,False,True,False,True,False,False,False


---
### Selección de Columnas

In [3]:
FEATURE_COLS = [col for col in train.columns if col != 'action_taken']
print(f"Nº de features: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")

Nº de features: 80
Features: ['derived_msa-md', 'county_code', 'preapproval', 'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage', 'open-end_line_of_credit', 'business_or_commercial_purpose', 'loan_amount', 'loan_term', 'negative_amortization', 'interest_only_payment', 'balloon_payment', 'other_nonamortizing_features', 'construction_method', 'occupancy_type', 'manufactured_home_secured_property_type', 'manufactured_home_land_property_interest', 'total_units', 'income', 'debt_to_income_ratio', 'applicant_credit_score_type', 'co-applicant_credit_score_type', 'applicant_ethnicity-1', 'co-applicant_ethnicity-1', 'applicant_ethnicity_observed', 'co-applicant_ethnicity_observed', 'applicant_race-1', 'co-applicant_race-1', 'applicant_race_observed', 'co-applicant_race_observed', 'applicant_sex', 'co-applicant_sex', 'applicant_sex_observed', 'co-applicant_sex_observed', 'applicant_age', 'applicant_age_above_62', 'submission_of_application', 'initially_payable_to_institution', 'aus-1

---
### Separación de Train/Test

In [4]:
X_train = train[FEATURE_COLS] # columnas que entran al modelo 
y_train = train['action_taken'] # columna que queremos predecir

X_test = test[FEATURE_COLS] 
y_test = test['action_taken'] 

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (226665, 80)
X_test:  (56667, 80)


---
---

## Comparativa de Modelos

Para abordar este problema de clasificación binaria, descartaremos algoritmos simples como los Árboles de Decisión por su tendencia al sobreajuste, y enfoques como Support Vector Machines (SVM) por su inasumible coste computacional frente a matrices dispersas de este tamaño. 

Nuestra estrategia se centrará exclusivamente en modelos de ensamble basados en árboles (estándar actual para datos tabulares). Para seleccionar el mejor modelo evaluaremos dos arquitecturas: por un lado, el paradigma *Bagging* (entrenamiento paralelo) mediante **Random Forest**, que reduce la varianza construyendo árboles independientes y promediando sus decisiones. Por otro lado, contrastaremos este enfoque con la familia *Boosting* (entrenamiento secuencial), donde cada nuevo árbol corrige iterativamente los errores del anterior. Dentro de esta segunda rama evaluaremos el modelo base de **Gradient Boosting** frente a sus evoluciones más avanzadas: **XGBoost**, que incorpora regularización matemática para un ajuste más robusto, y **LightGBM**, que optimiza la construcción de las ramas logrando una eficiencia computacional máxima en *datasets* voluminosos.

In [5]:
mejores_resultados = []

---

### Random Forest

In [6]:
resultados_rf = []

#### Modelo por defecto

In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

resultados_rf.append({
    'Modelo': 'RF por defecto',
    'Accuracy': round(accuracy_score(y_test, y_pred_rf), 4),
    'F1-Score': round(f1_score(y_test, y_pred_rf), 4),
    'ROC-AUC': round(roc_auc_score(y_test, y_prob_rf), 4)
})

print(f"Classification Report:\n")
print(classification_report(y_test, y_pred_rf))

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.56      0.67     14909
           1       0.86      0.96      0.91     41758

    accuracy                           0.86     56667
   macro avg       0.85      0.76      0.79     56667
weighted avg       0.85      0.86      0.85     56667



#### Optimización por GridSearch

In [8]:
param_grid = {
    'n_estimators':     [100, 200],
    'max_features':     ['sqrt', 'log2', None],
    'max_depth':        [None, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True),
    param_grid,
    cv=5,
    scoring='roc_auc',  # estándar de la industria financiera
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)


# Mejor modelo
y_pred_grid_rf = grid_rf.best_estimator_.predict(X_test)
y_prob_grid_rf = grid_rf.best_estimator_.predict_proba(X_test)[:, 1]

resultados_rf.append({
    'Modelo': 'RF por defecto',
    'Accuracy': round(accuracy_score(y_test, y_pred_grid_rf), 4),
    'F1-Score': round(f1_score(y_test, y_pred_grid_rf), 4),
    'ROC-AUC': round(roc_auc_score(y_test, y_prob_grid_rf), 4)
})

print(f"Mejores parámetros: {grid_rf.best_params_}\n")

print(f"Classification Report:\n")
print(classification_report(y_test, y_pred_grid_rf))

KeyboardInterrupt: 

#### Mejor modelo Random Forest

In [ ]:
comparacion_rf = pd.DataFrame(resultados_rf).set_index('Modelo')
display(comparacion_rf)

Seleccionamos como mejor modelo... (con x configuración)

In [ ]:
# Pasamos el mejor modelo de RF a la lista de mejores modelos generales
mejor_rf = comparacion_rf.loc[comparacion_rf['ROC-AUC'].idxmax()].to_dict()
mejores_resultados.append(mejor_rf)